# Replaying someone else's tape gets you 88% of their score

Almost everyone in this competition is running a fixed program. I measured it: on 26 real replays of my scoring submission, 25 of the 26 opponents were playing a tape.

So the obvious move is to find the strongest tape and play that one. I spent a while on it. Here is the number that ended the search, and I think it saves you the same detour.

**Replaying a tape reproduces about 88% of its owner's score.** Not 100. And that 12% gap is enough to put gold out of reach for every tape in the competition, including the one belonging to the team in first place.


## The measurement

Two teams, replayed in full, scored against the same opponents.

| tape replayed | fraction of the owner's score I recovered |
|---|---|
| tiki | **92.0%** |
| gogogo | **84.8%** |
| average of the two | **88.4%** |

My first guess was that the loss came from the reactive part. A tape is a fixed sequence, so anything the owner's agent did in response to what opponents were doing is gone when I replay it.

That guess was wrong, and the check is simple. I replayed gogogo's tape against gogogo, in both seats. Self-similarity 100.0%, both seats, and it still lost 15%.

So the missing 12% is not reactivity. Something else about being the tape's owner is worth 12%, and I could not recover it by copying harder.


## What that implies about the ceiling

Work it backwards.

Gold sits at 2746.3 (rank 24 of 8151, as of 8 September). To reach that by replay, at 88% recovery, you would need a tape whose owner scored:

```
2746.3 / 0.884 = 3106
```

The highest-scoring team in the whole competition, the leader is at **2914.5**.

There is no such tape. Copy the single best player in the competition and you land at roughly 2576, which is **170 points short of gold**.

The entire family of copy-the-tape strategies is capped below the medal line. That is not a statement about which tape to pick. There isn't one.


## Where I actually am, and why I stopped looking

I am at **2527.7, rank 271 of 8151**. Comfortable silver territory, 219 points off gold.

For a while I kept probing tapes, on the theory that a better one existed. The arithmetic above is what stopped me. The best possible outcome of that search was about 2576, and I was already close enough to it that the remaining upside was smaller than it looked.

That reframes the problem. Gold does not come from a better tape. It comes from an agent that generates both halves of the game itself, the farming and the market, and reacts.


## Two things worth knowing if you go that way

**96% of your opponents are tapes.** That is not a complaint, it is an exploitable fact. You are playing against fixed sequences that will not adapt to anything you do. An agent that notices what the field is doing has 25 of 26 opponents who cannot notice back.

**Score in coins, not in wins.** I nearly fooled myself here. A change can win more games and still lose money, because the losses get worse. Measure the thing the leaderboard measures.

And if you are benchmarking anything, pin the opponents. The engine draws shop stock from the same random stream as the weeds, so two runs with the same seed can still land in different towns and swing the result by twenty thousand coins. I lost a day to that before I pinned them.


## Work out your own ceiling

Plug in what you measure and it tells you whether copying can reach the score you want, and what recovery rate you would need if it can't.

The only input you have to measure yourself is the recovery rate: replay someone's tape, score it, divide by their score. Two replays is enough to get a working figure.


In [ ]:
def copy_ceiling(best_tape_score, recovery, target):
    """Can replaying tapes reach `target`?

    best_tape_score : the highest score any team you can copy actually holds
    recovery        : fraction of a tape owner's score your replay reproduces
    target          : the score you are aiming at, e.g. the gold cutoff
    """
    reachable = best_tape_score * recovery
    needed_owner = target / recovery
    needed_recovery = target / best_tape_score
    return {
        "best you can reach by copying": round(reachable, 1),
        "short of target by": round(target - reachable, 1),
        "owner score you would need": round(needed_owner, 1),
        "recovery you would need": round(needed_recovery, 3),
        "possible": reachable >= target,
    }


# Kaggriculture, 2026-09-01 standings
GOLD, BEST_TAPE = 2746.3, 2914.5   # public LB, 8 September
RECOVERY = (0.920 + 0.848) / 2          # the two replays I measured

for k, v in copy_ceiling(BEST_TAPE, RECOVERY, GOLD).items():
    print(f"  {k:32s} {v}")

print()
print("  where I am: 2527.7, rank 271 of 8151")
print("  so the whole copy-a-tape family tops out ~170 points under gold,")
print("  and no choice of tape changes that.")


## A stronger test than mine, from someone else

While writing this I found [Thomas Tschinkel's replay router](https://www.kaggle.com/code/thomastschinkel/kaggriculture-95-5-win-rate-via-replay-routing), which is a much better version of the copying idea than anything I built. It plays a six day opening, reads the town and the market, then splices into whichever public schedule fits what the seed actually offers. Over 44,096 held out games it wins 95.5% against replays, versus 84.6% for a single fixed replay.

The insight I liked most: a splice is only safe between replays that share an opening, because the incoming schedule inherits a farm it did not plant. He hashed the first six days of every replay and found 623 distinct openings, with one group covering 36% of his corpus. The single strongest replay he has sits in a group of size one and can therefore never route anywhere.

That is a real advance over playing one tape, and it is worth reading on its own.

It also happens to be the sharpest test of the arithmetic above, so here is the number. On the public leaderboard that router scores **2531.4**. My own submission, which does none of this, scores 2527.7. Gold is 2746.3.

So the most sophisticated public copying method available beats a single tape convincingly on win rate, and lands in the same place as everything else on score. It sits **below** the 2576 ceiling this notebook computes, not above it.

Win rate and score are not the same currency. Winning a game by a hundred coins and winning it by two thousand both count as one win.


## The short version

If you are copying tapes: the ceiling is 88% of the best tape available, which is 2585, which is 102 short of gold. Stop tuning which tape.

If you have a replay that recovers more than 92%, I would like to see it. That would break the arithmetic above, and it is the only thing that would.
